PyAnalytics - Resolução de Problemas de Análise
de Dados e Compartilhamento de Conhecimentos

Projeto Final - Esther de Oliveira - UFSC

1 - Introdução e Fonte de Dados

Eu escolhi trabalhar com o dataset de Ocupação de Leitos Hospitalares de 2022, extraído da base pública ESUS-VEPI. O meu objetivo foi realizar uma análise regional, focando exclusivamente nos dados de Araranguá/SC. Essa escolha me permitiu observar como o sistema de saúde local se comportou em um período de transição da pandemia, saindo de um início de ano crítico para um segundo semestre de maior controle.

2 - Importação de Bibliotecas e Dados

- Pandas: Eu utilizei como a biblioteca base para toda a manipulação do DataFrame e leitura do CSV.

- NumPy: Eu apliquei para realizar cálculos matemáticos de alta performance, especialmente na somatória de grandes volumes de dados.

- Matplotlib: Eu usei como a minha ferramenta de visualização para transformar as tabelas em gráficos de linha.

- Google Drive API: Eu realizei a carga dos dados diretamente de uma URL do Drive usando o ID do arquivo, garantindo que o meu projeto seja reprodutível em qualquer ambiente Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configurações de exibição do Pandas para tabelas mais limpas
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

# Carga e Filtragem Inicial
df_raw = pd.read_csv('esus-vepi.LeitoOcupacao_2022.csv', usecols=colunas_interesse)

# Carregamos apenas as colunas que realmente importam para economizar memória
colunas_interesse = [
    'dataNotificacao', 'municipio', 'ocupacaoCovidUti',
    'ocupacaoCovidCli', 'saidaConfirmadaObitos'
]

df_raw = pd.read_csv(url, usecols=colunas_interesse)

# Filtro e Limpeza em uma única etapa (Method Chaining)
df = (df_raw[df_raw['municipio'] == 'Araranguá']
      .copy()
      .assign(dataNotificacao=lambda x: pd.to_datetime(x['dataNotificacao']))
      .fillna(0)
      .sort_values('dataNotificacao'))

# Criar coluna de mês para agrupamento
df['mes'] = df['dataNotificacao'].dt.month

df.head()

: 

3 - Diagnóstico do Dataset

Eu realizei um "check-up" completo inicial para entender a qualidade da base de dados. Eu usei o info() e descobri que as datas estavam em formato de texto, o que impediria qualquer cálculo temporal. Além disso, eu executei o isnull().sum() e notei que muitas colunas hospitalares estavam vazias (NaN). Eu entendi que esses "buracos" precisavam ser tratados na limpeza, caso contrário, o meu gráfico de linha não seria gerado corretamente.

In [ ]:
# Verificar tipos de dados e colunas
print("Estrutura do Dataset:")
df.info()

# Verificar dados faltantes (importante para a próxima etapa de limpeza)
print("\nDados Faltantes por Coluna:")
print(df.isnull().sum())

4 - Tratamento e Saneamento

Eu executei procedimentos de limpeza essenciais para garantir a integridade da análise. Eu usei o pd.to_datetime para converter as datas e, o mais importante, utilizei o fillna(0) para substituir os valores nulos por zero. Eu fiz isso porque, na gestão hospitalar, se não houve registro de ocupação em um dia, a interpretação correta é que o hospital estava com zero pacientes, e não que o dado "sumiu". Por fim, eu usei o sort_values para colocar tudo em ordem cronológica, evitando que o gráfico ficasse bagunçado.

In [ ]:
# Transformar a coluna de data de 'texto' para 'datetime'
df['dataNotificacao'] = pd.to_datetime(df['dataNotificacao'])

# Ordenar os dados por data para não dar erro
df = df.sort_values(by='dataNotificacao')

# Tratar valores nulos
colunas_hospitalares = ['ocupacaoCovidUti', 'ocupacaoCovidCli', 'saidaConfirmadaObitos']
df[colunas_hospitalares] = df[colunas_hospitalares].fillna(0)

# Criar o DataFrame limpo
df_limpo = df[['dataNotificacao', 'ocupacaoCovidUti', 'ocupacaoCovidCli', 'saidaConfirmadaObitos']].copy()

df.head()

5 - Agregações e Estatísticas

- groupby(): Eu utilizei para agrupar centenas de registros diários em médias mensais, o que me permitiu ver a "fotografia" do ano de forma simplificada.

- sum(): Eu apliquei esta função para calcular o total acumulado de óbitos, dando uma dimensão real do impacto da saúde em Araranguá.

- dt.month: Eu usei para extrair o mês de cada data, criando a base necessária para o meu agrupamento.

- copy(): Eu utilizei para criar um DataFrame isolado para Araranguá, protegendo os dados originais de qualquer alteração acidental.

- sort_values(): Eu usei para garantir que o fluxo de dados seguisse a ordem correta do calendário de 2022.

In [ ]:
# Criar uma coluna de 'mês'
df_limpo['mes'] = df_limpo['dataNotificacao'].dt.month

# Groupby - Média mensal
media_mensal_uti = df_limpo.groupby('mes')['ocupacaoCovidUti'].mean()

# Usando Numpy
total_obitos = np.sum(df_limpo['saidaConfirmadaObitos'].values)

print(f"Total de óbitos em 2022 em Araranguá: {total_obitos}")
print("\nMédia de leitos UTI ocupados por mês:")
print(media_mensal_uti)

6 - Análise Visual de Tendências

Eu construí um gráfico de linha utilizando o Matplotlib para representar a pressão hospitalar na UTI de Araranguá. Eu escolhi o gráfico de linha porque ele é o mais eficaz para demonstrar tendências ao longo do tempo. Eu adicionei títulos, rótulos nos eixos e uma grade de fundo para facilitar a leitura. Notei que a linha cai para o zero no segundo semestre, o que reflete fielmente o que está na base de dados oficial.

In [ ]:
plt.style.use('seaborn-v0_8-muted') # Estilo visual mais moderno
fig, ax = plt.subplots(figsize=(12, 6))

# Plotagem com melhorias estéticas
ax.plot(df['dataNotificacao'], df['ocupacaoCovidUti'],
        label='UTI COVID', color='#d62728', linewidth=2, alpha=0.8)

ax.fill_between(df['dataNotificacao'], df['ocupacaoCovidUti'], color='#d62728', alpha=0.1)

# Customização técnica
ax.set_title('Evolução da Pressão em UTI COVID - Araranguá', fontsize=15, pad=20, fontweight='bold')
ax.set_xlabel('Mês de Referência', fontsize=12)
ax.set_ylabel('Nº de Pacientes', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(frameon=True, facecolor='white')

# Remover bordas desnecessárias (spines)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

Eu finalizei a minha análise com insights bem claros e completos. Eu observei que Araranguá teve um início de 2022 com ocupação significativa, mas a partir de agosto (mês 8) até novembro (mês 11), os valores caíram para zero. Isso indica que, nesse período específico, os hospitais da cidade não registraram internações por COVID-19 nas UTIs mapeadas, demonstrando o sucesso das medidas de controle e vacinação. No lado técnico, eu comprovei que a análise de dados requer atenção aos detalhes: o que parecia um erro (os zeros) era na verdade a informação mais importante sobre a recuperação da saúde pública na região.

In [ ]:
total_obitos = int(np.sum(df['saidaConfirmadaObitos']))
pico_uti = int(df['ocupacaoCovidUti'].max())

print(f"{'='*30}")
print(f"ESTATÍSTICAS FINAIS 2022")
print(f"{'='*30}")
print(f"Total de Óbitos: {total_obitos}")
print(f"Pico de Ocupação UTI: {pico_uti} pacientes")
print(f"{'='*30}")